# Fintel Run Analytics

Swap config below and re-run.

In [ ]:
import sys
from pathlib import Path

# ── config ──────────────────────────────────────────────────────────
# Resolve relative to the fintel repo root (parent of fintel/).
FINTEL_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fintel" / "evaluate").is_dir()
)
JOB_DIR = FINTEL_ROOT / "runs" / "djia-w-full-r1-0001"

HORIZONS = [1, 2, 4, 8]
LONG_THRESHOLDS = [0.0, 0.5]  # long-basket cutoffs
ACTIVE_BUDGET = 0.5  # naive tilt + MVO
COST_BPS = 5.0
RISK_AVERSION = 2.0  # MVO λ
COV_LOOKBACK = 60  # trading days for Σ
ATTR_FOCUS = None  # e.g. "MVO"; None → first non-benchmark
ATTR_TOP_N = 8

assert JOB_DIR.is_dir(), JOB_DIR
if str(FINTEL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINTEL_ROOT))
print(JOB_DIR)

## 1. Decisions → DataFrame

In [ ]:
import json
from datetime import date as Date

import matplotlib.pyplot as plt
import pandas as pd

rows = []
for run_dir in sorted(JOB_DIR.glob("r*")):
    if not run_dir.name[1:].isdigit():
        continue
    k = int(run_dir.name[1:])
    for trial_dir in sorted((run_dir / "trials").iterdir()):
        if not trial_dir.is_dir():
            continue
        dp = trial_dir / "decision.json"
        if not dp.is_file():
            continue
        d = Date.fromisoformat(trial_dir.name)
        for sym, v in json.loads(dp.read_text()).items():
            rows.append(
                {
                    "run": run_dir.name,
                    "k": k,
                    "decision_date": d,
                    "symbol": sym,
                    "score": v.get("score"),
                    "conviction": v.get("conviction"),
                    "time_horizon": v.get("time_horizon"),
                    "rationale": v.get("rationale", ""),
                    "n_key_factors": len(v.get("key_factors", [])),
                    "n_sources": len(v.get("sources_cited", [])),
                }
            )
df = pd.DataFrame(rows)
print(
    f"{len(df)} rows, {df['symbol'].nunique()} symbols, {df['run'].nunique()} run(s), {df['decision_date'].nunique()} dates"
)
df.head()

## 2. Stochasticity (K > 1 only)

In [ ]:
from scipy.stats import spearmanr

k_repeats = df["k"].nunique()
if k_repeats < 2:
    print(f"k_repeats={k_repeats} — skip")
else:
    pivot = df.pivot_table(index=["decision_date", "symbol"], columns="run", values="score")
    cols = pivot.columns.tolist()
    print("Pairwise Spearman rank corr:")
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            corrs = []
            for _, g in pivot.groupby(level=0):
                a, b = g[cols[i]].dropna(), g[cols[j]].dropna()
                common = a.index.intersection(b.index)
                if len(common) >= 2:
                    corrs.append(spearmanr(a.loc[common], b.loc[common]).correlation)
            print(f"  {cols[i]} vs {cols[j]}: mean={sum(corrs) / len(corrs):.3f} (n={len(corrs)})")
    print("\nTop 10 by cross-run score std:")
    print(pivot.groupby(level=1).std().mean(axis=1).sort_values(ascending=False).head(10).round(4))

## 3. Score Distributions Over Time

In [ ]:
import math

ens = df.copy()
if k_repeats > 1:
    ens = ens.groupby(["decision_date", "symbol"])["score"].mean().reset_index()
pivot_score = ens.pivot_table(index="decision_date", columns="symbol", values="score")
syms = sorted(pivot_score.columns)
ncols = 5
nrows = math.ceil(len(syms) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 2.4 * nrows), sharex=True)
axes = axes.flatten()
for ax, sym in zip(axes, syms):
    s = pivot_score[sym].dropna()
    ax.plot(s.index, s.values, marker="o", markersize=3, linewidth=1)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title(f"{sym} (n={len(s)})", fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes[len(syms) :]:
    ax.axis("off")
fig.suptitle(f"Score by ticker ({len(syms)} names)", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

df.score.plot(kind="hist", bins=100, title="Score histogram")
plt.show()

## 4. KPI Metrics

In [ ]:
from fintel.evaluate.kpi import _forward_returns, _pearson, _spearman_ic
from fintel.evaluate.prices import price_lookup_for
from fintel.evaluate.read import load_job
from fintel.evaluate.signals import build_signals
from fintel.models.strategy import ScoringSpec

runs = load_job(JOB_DIR)
cfg = json.loads((JOB_DIR / "r1" / "config.json").read_text())
scoring = ScoringSpec.model_validate(cfg["scoring"])
signals = build_signals(runs, signal=scoring.signal, transform=scoring.transform)
prices = price_lookup_for(JOB_DIR)

dates = sorted(signals.ensemble.keys())
universe = sorted(signals.universe)


def _icir(vals):
    if len(vals) < 2:
        return None
    mu = sum(vals) / len(vals)
    std = (sum((v - mu) ** 2 for v in vals) / (len(vals) - 1)) ** 0.5
    return mu / std if std > 0 else None


def _pearson_ic(signal, fwd):
    common = sorted(set(signal) & set(fwd))
    if len(common) < 2:
        return None
    return _pearson([signal[s] for s in common], [fwd[s] for s in common])


def ic_by_horizon(kind):
    fn = _spearman_ic if kind == "spearman" else _pearson_ic
    out = {}
    for h in HORIZONS:
        fwd = _forward_returns(dates, universe, prices, h)
        vals, ics = [], []
        for d in dates:
            if d not in fwd:
                continue
            ic = fn(signals.ensemble[d], fwd[d])
            if ic is not None:
                vals.append(ic)
                ics.append((d, ic))
        out[h] = {
            "mean_ic": sum(vals) / len(vals) if vals else None,
            "raw_icir": _icir(vals),
            "ic_values": ics,
            "n_periods": len(vals),
        }
    return out


spearman = ic_by_horizon("spearman")
pearson = ic_by_horizon("pearson")

print(f"{'h':>3}  {'sp_mean':>9}  {'sp_icir':>8}  {'pe_mean':>9}  {'pe_icir':>8}  {'n':>3}")
for h in HORIZONS:
    s, p = spearman[h], pearson[h]
    print(
        f"{h:>3}  {s['mean_ic'] or 0:>9.4f}  {s['raw_icir'] or 0:>8.4f}  "
        f"{p['mean_ic'] or 0:>9.4f}  {p['raw_icir'] or 0:>8.4f}  {s['n_periods']:>3}"
    )

h0 = HORIZONS[0]
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, series, title in [
    (axes[0], spearman[h0]["ic_values"], f"Spearman IC (h={h0})"),
    (axes[1], pearson[h0]["ic_values"], f"Pearson IC (h={h0})"),
]:
    vals = [v for _, v in series]
    ax.bar(range(len(vals)), vals, color=["g" if v >= 0 else "r" for v in vals])
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([d.isoformat() for d, _ in series], rotation=45, fontsize=7)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 5. Performance

Benchmark = price-weighted DJIA. Long baskets from `LONG_THRESHOLDS`. Empty basket → cash (r=0).

In [ ]:
from datetime import timedelta

import numpy as np

from fintel.market.calendar import TradingCalendar

cal = TradingCalendar()


def price_weighted_weights(d):
    px = {s: prices.price_at(s, d) for s in signals.ensemble.get(d, {})}
    px = {s: p for s, p in px.items() if p and p > 0}
    total = sum(px.values())
    return {s: p / total for s, p in px.items()}


def long_names(d, threshold):
    return {s: v for s, v in signals.ensemble.get(d, {}).items() if v > threshold}


def score_weighted_long(d, threshold):
    longs = long_names(d, threshold)
    if not longs:
        return {}
    total = sum(longs.values())
    return {s: v / total for s, v in longs.items()}


def equal_weight_long(d, threshold):
    longs = long_names(d, threshold)
    if not longs:
        return {}
    w = 1.0 / len(longs)
    return {s: w for s in longs}


def naive_score_tilt(d, active_budget=ACTIVE_BUDGET):
    sig = signals.ensemble.get(d, {})
    if not sig:
        return {}
    n = len(sig)
    total_abs = sum(abs(v) for v in sig.values()) or 1.0
    raw = {s: max(0.0, 1.0 / n + active_budget * v / total_abs) for s, v in sig.items()}
    tw = sum(raw.values()) or 1.0
    return {s: w / tw for s, w in raw.items()}


def _prev_trading_day(d):
    for i in range(1, 10):
        d2 = d - timedelta(days=i)
        if cal.is_trading_day(d2):
            return d2
    return None


def cov_matrix(d, lookback=COV_LOOKBACK):
    px = {}
    for s in universe:
        series, cur, steps = [], d, 0
        while steps < lookback + 1 and cur is not None:
            p = prices.price_at(s, cur)
            if p is not None:
                series.append((cur, p))
            cur = _prev_trading_day(cur)
            steps += 1
        if len(series) >= 20:
            px[s] = dict(series)
    if len(px) < 2:
        return None
    all_dates = sorted({dt for m in px.values() for dt in m})[-lookback:]
    pdf = pd.DataFrame({s: [m.get(dt) for dt in all_dates] for s, m in px.items()}, index=all_dates)
    pdf = pdf.ffill().dropna(axis=1, how="all")
    rets = pdf.pct_change().dropna()
    if len(rets) < 5 or pdf.shape[1] < 2:
        return None
    return rets.cov() * 252


def mvo_weights(d, active_budget=ACTIVE_BUDGET, risk_aversion=RISK_AVERSION):
    import cvxpy as cp

    sig = signals.ensemble.get(d, {})
    wbm = price_weighted_weights(d)
    sigma = cov_matrix(d)
    if sigma is None:
        return naive_score_tilt(d, active_budget)
    common = sorted(set(sig) & set(sigma.index) & set(wbm))
    if len(common) < 2:
        return wbm
    mu = np.array([sig[s] for s in common], dtype=float)
    mu = mu - mu.mean()
    wbm_v = np.array([wbm[s] for s in common], dtype=float)
    Sig = sigma.loc[common, common].values.astype(float)
    n = len(common)
    cap = max(active_budget / n, 0.01)
    w = cp.Variable(n)
    active = w - wbm_v
    obj = cp.Maximize(mu @ w - 0.5 * risk_aversion * cp.quad_form(w, cp.psd_wrap(Sig)))
    cons = [
        cp.sum(w) == float(wbm_v.sum()),
        cp.sum(active) == 0,
        cp.abs(active) <= cap,
        cp.norm1(active) <= 2.0 * active_budget,
    ]
    prob = cp.Problem(obj, cons)
    prob.solve()
    if w.value is None or prob.status not in ("optimal", "optimal_inaccurate"):
        return naive_score_tilt(d, active_budget)
    w_full = np.where(np.abs(w.value) < 1e-9, 0.0, np.asarray(w.value).flatten())
    s = w_full.sum()
    if abs(s) > 1e-9 and abs(s - wbm_v.sum()) > 1e-9:
        w_full = w_full * (wbm_v.sum() / s)
    out = dict(wbm)
    for i, sym in enumerate(common):
        out[sym] = float(w_full[i])
    return out


def nav_series(weights_fn, cost_bps=COST_BPS):
    nav_g = nav_n = 1.0
    prev_w = {}
    start = dates[0].isoformat()
    gross = [{"date": start, "nav": 1.0}]
    net = [{"date": start, "nav": 1.0}]
    for i, d in enumerate(dates[:-1]):
        end = dates[i + 1]
        w = weights_fn(d) or {}
        if not w:
            r = 0.0
        else:
            fwd = {
                s: fr
                for s, fr in ((s, prices.forward_return(s, d, end)) for s in w)
                if fr is not None
            }
            if not fwd:
                r = 0.0
            else:
                mass = sum(w[s] for s in fwd) or 1.0
                r = sum((w[s] / mass) * fwd[s] for s in fwd)
        turnover = sum(abs(w.get(s, 0.0) - prev_w.get(s, 0.0)) for s in set(w) | set(prev_w))
        cost = 0.0 if i == 0 else turnover * cost_bps / 10000.0
        nav_g *= 1.0 + r
        nav_n *= 1.0 + r - cost
        gross.append({"date": end.isoformat(), "nav": round(nav_g, 6)})
        net.append({"date": end.isoformat(), "nav": round(nav_n, 6)})
        prev_w = dict(w)
    return {"gross": gross, "net": net}


# Build strategy book from config — labels + weight fns
strategies = {"DJIA PW": price_weighted_weights}
for thr in LONG_THRESHOLDS:
    strategies[f"SW Long >{thr}"] = lambda d, t=thr: score_weighted_long(d, t)
    strategies[f"EW Long >{thr}"] = lambda d, t=thr: equal_weight_long(d, t)
strategies["Naive tilt"] = naive_score_tilt
strategies["MVO"] = mvo_weights

results = {label: nav_series(fn) for label, fn in strategies.items()}

fig, ax = plt.subplots(figsize=(11, 5))
for label, res in results.items():
    xs = [r["date"] for r in res["net"]]
    ys = [r["nav"] for r in res["net"]]
    ax.plot(xs, ys, marker="o", markersize=3, linewidth=1.4, label=label)
ax.set_title(f"Net NAV (cost={COST_BPS}bps)")
ax.set_ylabel("NAV")
ax.legend(fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Final NAV (net):")
for label, res in results.items():
    print(f"  {label:<22} {res['net'][-1]['nav']:.4f}")

## 6. Attribution

Name active contribution vs benchmark: `(w_strat − w_bench) · r` per period, summed. Gross (pre-cost).

In [ ]:
bench_label = next(iter(strategies))
bench_fn = strategies[bench_label]
attr_strategies = {k: v for k, v in strategies.items() if k != bench_label}


def attribute_vs_bench(weights_fn):
    contrib, avg_active, period_rows = {}, {}, []
    for i, d in enumerate(dates[:-1]):
        end = dates[i + 1]
        wb = bench_fn(d) or {}
        ws = weights_fn(d) or {}
        names = set(wb) | set(ws)
        fwd = {
            s: fr
            for s, fr in ((s, prices.forward_return(s, d, end)) for s in names)
            if fr is not None
        }

        def port_ret(w):
            if not w:
                return 0.0, {}
            common = [s for s in w if s in fwd]
            if not common:
                return 0.0, {}
            mass = sum(w[s] for s in common) or 1.0
            wn = {s: w[s] / mass for s in common}
            return sum(wn[s] * fwd[s] for s in common), wn

        r_s, wn_s = port_ret(ws)
        r_b, wn_b = port_ret(wb)
        period_sum = 0.0
        for s in set(wn_s) | set(wn_b):
            wa = wn_s.get(s, 0.0) - wn_b.get(s, 0.0)
            c = wa * fwd[s]
            contrib[s] = contrib.get(s, 0.0) + c
            avg_active.setdefault(s, []).append(wa)
            period_sum += c
        active = r_s - r_b
        period_rows.append(
            {
                "start": d,
                "end": end,
                "r_strat": r_s,
                "r_bench": r_b,
                "active": active,
                "sum_contrib": period_sum,
                "residual": active - period_sum,
            }
        )
    return {
        "contrib": contrib,
        "avg_active": {s: sum(v) / len(v) for s, v in avg_active.items()},
        "periods": pd.DataFrame(period_rows),
        "sum_contrib": sum(contrib.values()),
        "sum_residual": float(pd.DataFrame(period_rows)["residual"].sum()),
        "arith_active": float(pd.DataFrame(period_rows)["active"].sum()),
    }


attr = {label: attribute_vs_bench(fn) for label, fn in attr_strategies.items()}

summary = pd.DataFrame(
    [
        {
            "strategy": label,
            "arith_active": a["arith_active"],
            "sum_contrib": a["sum_contrib"],
            "residual": a["sum_residual"],
            "top_name": max(a["contrib"], key=a["contrib"].get),
            "top_contrib": max(a["contrib"].values()),
            "bottom_name": min(a["contrib"], key=a["contrib"].get),
            "bottom_contrib": min(a["contrib"].values()),
        }
        for label, a in attr.items()
    ]
).set_index("strategy")
print(f"Active vs {bench_label} (gross, arithmetic)")
summary.round(4)

In [ ]:
n_strat = len(attr)
ncols = 3
nrows = math.ceil(n_strat / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.atleast_1d(axes).flatten()
for ax, (label, a) in zip(axes, attr.items()):
    s = pd.Series(a["contrib"]).sort_values()
    half = ATTR_TOP_N // 2
    top = pd.concat([s.head(half), s.tail(half)])
    ax.barh(top.index, top.values, color=["#c0392b" if v < 0 else "#27ae60" for v in top.values])
    ax.axvline(0, color="gray", linewidth=0.6)
    ax.set_title(f"{label}\narith active={a['arith_active']:+.3%}", fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes[n_strat:]:
    ax.axis("off")
fig.suptitle(f"Name active contribution vs {bench_label}", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

focus = ATTR_FOCUS or next(iter(attr))
a = attr[focus]
detail = pd.DataFrame({"contrib": a["contrib"], "avg_active_w": a["avg_active"]})
detail = detail.sort_values("contrib", ascending=False)
detail["contrib_bps"] = detail["contrib"] * 1e4
print(f"\n{focus}: name attribution")
detail.round(4)